In [ ]:
import os
import pandas as pd

In [ ]:
RAW_PATH = "Fraud_Detection_Project/data/raw"
PROCESSED_PATH = "Fraud_Detection_Project/data/processed"

In [ ]:
os.makedirs(PROCESSED_PATH, exist_ok=True)

In [ ]:
users = pd.read_csv(os.path.join(RAW_PATH, "sd254_users.csv"))
cards = pd.read_csv(os.path.join(RAW_PATH, "sd254_cards.csv"))
transactions = pd.read_csv(os.path.join(RAW_PATH, "transactions_sample.csv"))

In [ ]:
# ============================================================
# TRANSACTION SAMPLE VALIDATION
# ============================================================
# The new transactions_sample.csv contains exactly 200,000
# transactions and preserves all 29,757 fraud cases from the
# original IBM transaction dataset.

EXPECTED_TRANSACTION_ROWS = 200000
EXPECTED_FRAUD_ROWS = 29757

if len(transactions) != EXPECTED_TRANSACTION_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_TRANSACTION_ROWS:,} transaction rows, "
        f"but found {len(transactions):,}."
    )

if "Is Fraud?" not in transactions.columns:
    raise ValueError("The transaction dataset must contain the 'Is Fraud?' column.")


In [ ]:
fraud_count = (
    transactions["Is Fraud?"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("yes")
    .sum()
)

if fraud_count != EXPECTED_FRAUD_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_FRAUD_ROWS:,} fraud transactions, "
        f"but found {fraud_count:,}."
    )

print("Transaction sample validation passed.")
print(f"Transactions : {len(transactions):,}")
print(f"Fraud        : {fraud_count:,}")
print(f"Fraud rate   : {fraud_count / len(transactions) * 100:.3f}%")

Transaction sample validation passed.
Transactions : 200,000
Fraud        : 29,757
Fraud rate   : 14.879%


In [ ]:
users = users.drop(columns=["Apartment","Address","Latitude","Longitude","Num Credit Cards"])

users.columns = (
    users.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

users.insert(0, "user_id", range(1, len(users) + 1))

In [ ]:
assert cards["User"].min() >= 0
assert cards["User"].max() < len(users)

assert transactions["User"].min() >= 0
assert transactions["User"].max() < len(users)

In [ ]:
# Split expiration date
cards[["expire_month", "expire_year"]] = (cards["Expires"].astype("string").str.split("/", expand=True))

# Split account opening date
cards[["opening_month", "opening_year"]] = (cards["Acct Open Date"].astype("string").str.split("/", expand=True))

# Remove original combined date columns
cards = cards.drop(columns=["Expires","Acct Open Date"])

cards.columns = (
    cards.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

cards.insert( 0, "card_id", range(1, len(cards) + 1))

print("Number of users:", len(users))

print("Cards User range:")
print(cards["user"].min(), cards["user"].max())

print("Transactions User range:")
print(transactions["User"].min(), transactions["User"].max())

cards["user_id"] = cards["user"] + 1

cards = cards.drop(columns=["user"])

Number of users: 2000
Cards User range:
0 1999
Transactions User range:
0 1999


In [ ]:
transactions.columns = (
    transactions.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("?", "")
    .str.replace("-", "_")
)

# Normalize the fraud flag for downstream SQL/analytics.
# "Is Fraud?" becomes "is_fraud", with consistent Yes/No values.
transactions["is_fraud"] = (
    transactions["is_fraud"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "yes": "Yes",
        "no": "No"
    })
)

if transactions["is_fraud"].isna().any():
    raise ValueError(
        "Unexpected values found in 'is_fraud'. "
        "Expected only 'Yes' or 'No'."
    )

transactions["merchant_state"] = (transactions["merchant_state"].fillna("Unknown"))
transactions["zip"] = (transactions["zip"].astype("string").fillna("Unknown"))
transactions["errors"] = (transactions["errors"] .fillna("No Error"))

In [ ]:
merchant_columns = ["merchant_name","merchant_city","merchant_state","zip","mcc"]

merchants = (transactions[merchant_columns].drop_duplicates().reset_index(drop=True))

merchants.insert(0,"merchant_id",range(1, len(merchants) + 1))

In [ ]:
transactions = transactions.merge(merchants,on=merchant_columns,how="left",validate="many_to_one")

assert transactions["merchant_id"].notna().all()

card_mapping = cards[[
        "card_id",
        "user_id",
        "card_index"
    ]].copy()

# The transaction's user is zero-based,
# while user_id is one-based.

card_mapping["user"] = (card_mapping["user_id"] - 1)

In [ ]:
transactions = transactions.merge(
    card_mapping[[
            "card_id",
            "user",
            "card_index"
        ]],left_on=["user", "card"],right_on=["user", "card_index"],how="left",validate="many_to_one")

# Check that every transaction received card_id.
assert transactions["card_id"].notna().all()

transactions.insert(0,"transaction_id",range(1, len(transactions) + 1))

transactions = transactions.drop(
    columns=[
        "user",
        "card",
        "merchant_name",
        "merchant_city",
        "merchant_state",
        "zip",
        "mcc",
        "card_index"
    ])

transactions = transactions[[
        "transaction_id",
        "card_id",
        "merchant_id",
        "year",
        "month",
        "day",
        "time",
        "amount",
        "use_chip",
        "errors",
        "is_fraud"
    ]]

In [ ]:
users = users.rename(
    columns={
        "per_capita_income___zipcode":"per_capita_income_zipcode",

        "yearly_income___person":"yearly_income_person"
    }
)

users["zipcode"] = (users["zipcode"].astype("string"))

In [ ]:
users["per_capita_income_zipcode"] = (
    users["per_capita_income_zipcode"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

users["yearly_income_person"] = (
    users["yearly_income_person"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

users["total_debt"] = (
    users["total_debt"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [ ]:
cards["credit_limit"] = (
    cards["credit_limit"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

cards = cards.drop(columns=["cvv"])

cards["card_last4"] = (cards["card_number"].astype("string").str[-4:])
cards["card_last4"] = (cards["card_last4"].astype("string").str.zfill(4))

cards = cards.drop("card_number",axis=1)

In [ ]:
merchants = merchants.rename(
    columns={
        "merchant_name": "merchant_source_id"
    }
)

merchants["merchant_source_id"] = (merchants["merchant_source_id"].astype("string"))

transactions["amount"] = (
    transactions["amount"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [ ]:
# Final fraud-count validation after preprocessing.
# Preprocessing must not lose or change any fraud transactions.
assert transactions["is_fraud"].eq("Yes").sum() == EXPECTED_FRAUD_ROWS

assert users["user_id"].notna().all()
assert users["user_id"].is_unique

assert cards["card_id"].notna().all()
assert cards["card_id"].is_unique

assert merchants["merchant_id"].notna().all()
assert merchants["merchant_id"].is_unique

assert transactions["transaction_id"].notna().all()
assert transactions["transaction_id"].is_unique

# Cards → Users
assert cards["user_id"].isin(users["user_id"]).all()

# Transactions → Cards
assert transactions["card_id"].isin(cards["card_id"]).all()


# Transactions → Merchants
assert transactions["merchant_id"].isin(merchants["merchant_id"]).all()

In [ ]:

print("NULL VALUES")

print("\nUsers:")
print(users.isnull().sum())

print("\nCards:")
print(cards.isnull().sum())

print("\nMerchants:")
print(merchants.isnull().sum())

print("\nTransactions:")
print(transactions.isnull().sum())

NULL VALUES

Users:
user_id                      0
person                       0
current_age                  0
retirement_age               0
birth_year                   0
birth_month                  0
gender                       0
city                         0
state                        0
zipcode                      0
per_capita_income_zipcode    0
yearly_income_person         0
total_debt                   0
fico_score                   0
dtype: int64

Cards:
card_id                  0
card_index               0
card_brand               0
card_type                0
has_chip                 0
cards_issued             0
credit_limit             0
year_pin_last_changed    0
card_on_dark_web         0
expire_month             0
expire_year              0
opening_month            0
opening_year             0
user_id                  0
card_last4               0
dtype: int64

Merchants:
merchant_id           0
merchant_source_id    0
merchant_city         0
merchant_state        0

In [ ]:
print("DATA TYPES")

print("\nUsers:")
print(users.dtypes)

print("\nCards:")
print(cards.dtypes)

print("\nMerchants:")
print(merchants.dtypes)

print("\nTransactions:")
print(transactions.dtypes)

DATA TYPES

Users:
user_id                               int64
person                               object
current_age                           int64
retirement_age                        int64
birth_year                            int64
birth_month                           int64
gender                               object
city                                 object
state                                object
zipcode                      string[python]
per_capita_income_zipcode           float64
yearly_income_person                float64
total_debt                          float64
fico_score                            int64
dtype: object

Cards:
card_id                           int64
card_index                        int64
card_brand                       object
card_type                        object
has_chip                         object
cards_issued                      int64
credit_limit                    float64
year_pin_last_changed             int64
card_on_dark_web       

In [ ]:
users.to_csv(os.path.join(PROCESSED_PATH,"users_clean.csv"),index=False)

cards.to_csv(os.path.join(PROCESSED_PATH,"cards_clean.csv"),index=False)

merchants.to_csv(os.path.join(PROCESSED_PATH,"merchants_clean.csv"),index=False)

transactions.to_csv(os.path.join(PROCESSED_PATH,"transactions_clean.csv"),index=False)

In [ ]:
print("FINAL PHASE 3 CLEANUP COMPLETED")

print("\nUsers:", users.shape)
print("Cards:", cards.shape)
print("Merchants:", merchants.shape)
print("Transactions:", transactions.shape)

print("\nFinal files saved successfully.")

FINAL PHASE 3 CLEANUP COMPLETED

Users: (2000, 14)
Cards: (6146, 15)
Merchants: (44459, 6)
Transactions: (200000, 11)

Final files saved successfully.
